In [1]:
import json
import uuid
from sqlalchemy import create_engine

from utils import reset_db, get_session, model_to_dict
from data.models import udahub

# Udahub Application

## Core Database

**Init DB**

In [2]:
udahub_db = "data/core/udahub.db"

In [3]:
reset_db(udahub_db)

2026-03-11 18:04:09,991 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-03-11 18:04:09,992 INFO sqlalchemy.engine.Engine COMMIT
✅ Recreated data/core/udahub.db with fresh schema


In [3]:
engine = create_engine(f"sqlite:///{udahub_db}", echo=False)
udahub.Base.metadata.create_all(bind=engine)

**Account**

In [4]:
account_id = "cultpass"
account_name = "CultPass Card"

In [6]:
with get_session(engine) as session:
    account = udahub.Account(
        account_id=account_id,
        account_name=account_name,
    )
    session.add(account)

## Integrations

**Knowledge Base**

In [ ]:
# DONE: Create additional 10 articles

In [7]:
cultpass_articles = []

with open('data/external/cultpass_articles.jsonl', 'r', encoding='utf-8') as f:
    for line in f:
        cultpass_articles.append(json.loads(line))

In [8]:
cultpass_articles

[{'title': 'How to Reserve a Spot for an Event',
  'content': 'If a user asks how to reserve an event:\n\n- Guide them to the CultPass app\n- Instruct them to browse the experience catalog and tap \'Reserve\'\n- If it\'s a premium or limited event, check if reservation confirmation is required via email\n- Remind them to arrive at least 15 minutes early with their QR code visible\n\n**Suggested phrasing:**\n"You can reserve an experience by opening the CultPass app, selecting your desired event, and tapping \'Reserve\'. Be sure to arrive 15 minutes early with your QR code ready."',
  'tags': 'reservation, events, booking, attendance'},
 {'title': "What's Included in a CultPass Subscription",
  'content': 'Each user is entitled to 4 cultural experiences per month, which may include:\n- Art exhibitions\n- Museum entries\n- Music concerts\n- Film screenings and more\n\nSome premium experiences may require an additional fee (visible in the app).\n\n**Suggested phrasing:**\n"Your CultPass s

In [9]:
if len(cultpass_articles) < 14:
    raise AssertionError("You should load the articles with at least 14 records")

In [10]:
with get_session(engine) as session:
    kb = []
    for article in cultpass_articles:
        knowledge = udahub.Knowledge(
            article_id=str(uuid.uuid4()),
            account_id=account_id,
            title=article["title"],
            content=article["content"],
            tags=article["tags"]
        )
        kb.append(knowledge)
    session.add_all(kb)     

In [11]:
with get_session(engine) as session:
    articles = session.query(udahub.Knowledge).all()
    for article in articles:
        print(article)

<Knowledge(article_id='ef67d5f7-2d3b-4a6b-bcfe-bc65752f9860', title='How to Reserve a Spot for an Event')>
<Knowledge(article_id='f85fcf5e-2960-4194-9a5f-ee40fb81899f', title='What's Included in a CultPass Subscription')>
<Knowledge(article_id='9dd607e1-f1c6-42b0-9d20-e6b20cbd8fb4', title='How to Cancel or Pause a Subscription')>
<Knowledge(article_id='ce7d33c7-8829-4e7c-ac7f-983d4c290f34', title='How to Handle Login Issues?')>
<Knowledge(article_id='e08766f3-2d65-4ed0-8bee-a74b4cad26bf', title='How to Request a Refund')>
<Knowledge(article_id='46fbd95f-6f16-4204-b4b2-729addd2dbf4', title='How to Update Profile Information')>
<Knowledge(article_id='2fc9eb82-0d55-4552-8846-43e689569a98', title='How to Delete a CultPass Account')>
<Knowledge(article_id='c63c2648-a570-40a9-8e26-927f4ff79d97', title='Troubleshooting App Crashes')>
<Knowledge(article_id='afb8d3c9-c87d-44fd-aaee-31bd127dc1e6', title='QR Code Not Displaying or Scanning')>
<Knowledge(article_id='596104f7-4205-490f-a152-ec239ba

**Ticket**

In [5]:
cultpass_users = []

with open('data/external/cultpass_users.jsonl', 'r', encoding='utf-8') as f:
    for line in f:
        cultpass_users.append(json.loads(line))

In [13]:
ticket_info = {
    "status": "open",
    "content": "I can't log in to my Cultpass account.",
    "owner_id": cultpass_users[0]["id"],
    "owner_name": cultpass_users[0]["name"],
    "role": "user",
    "channel": "chat",
    "tags": "login, access",
}

In [14]:
with get_session(engine) as session:
    user = session.query(udahub.User).filter_by(
        account_id=account_id,
        external_user_id=ticket_info["owner_id"],
    ).first()

    if not user:
        user = udahub.User(
            user_id=str(uuid.uuid4()),
            account_id=account_id,
            external_user_id=ticket_info["owner_id"],
            user_name=ticket_info["owner_name"],
        )
    
    ticket = udahub.Ticket(
        ticket_id=str(uuid.uuid4()),
        account_id=account_id,
        user_id=user.user_id,
        channel=ticket_info["channel"],
    )
    metadata = udahub.TicketMetadata(
        ticket_id=ticket.ticket_id,
        status=ticket_info["status"],
        main_issue_type=None,
        tags=ticket_info["tags"],
    )

    first_message = udahub.TicketMessage(
        message_id=str(uuid.uuid4()),
        ticket_id=ticket.ticket_id,
        role=ticket_info["role"],
        content=ticket_info["content"],
    )

    session.add_all([user, ticket, metadata, first_message])


**Register all CultPass users in UdaHub**

Every CultPass customer needs a corresponding UdaHub user record so the support system can create tickets, track history, and escalate on their behalf.

In [6]:
with get_session(engine) as session:
    for cp_user in cultpass_users:
        # Skip if already registered
        existing = session.query(udahub.User).filter_by(
            account_id=account_id,
            external_user_id=cp_user["id"],
        ).first()
        if not existing:
            user = udahub.User(
                user_id=str(uuid.uuid4()),
                account_id=account_id,
                external_user_id=cp_user["id"],
                user_name=cp_user["name"],
            )
            session.add(user)
            print(f"  Added: {cp_user['name']} ({cp_user['id']})")
        else:
            print(f"  Exists: {cp_user['name']} ({cp_user['id']})")

# Verify
with get_session(engine) as session:
    users = session.query(udahub.User).all()
    print(f"\nTotal UdaHub users: {len(users)}")
    for u in users:
        print(f"  {u.user_name} | external_id={u.external_user_id}")

  Exists: Alice Kingsley (a4ab87)
  Added: Bob Stone (f556c0)
  Added: Cathy Bloom (88382b)
  Added: David Noir (888fb2)
  Added: Eva Green (f1f10d)
  Added: Frank Ocean (e6376d)

Total UdaHub users: 6
  Alice Kingsley | external_id=a4ab87
  Bob Stone | external_id=f556c0
  Cathy Bloom | external_id=88382b
  David Noir | external_id=888fb2
  Eva Green | external_id=f1f10d
  Frank Ocean | external_id=e6376d


# Tests

In [15]:
with get_session(engine) as session:
    account = session.query(udahub.Account).filter_by(
        account_id=account_id
    ).first()
    print(account)

<Account(account_id='cultpass', account_name='CultPass Card')>


In [16]:
with get_session(engine) as session:
    account = session.query(udahub.Account).filter_by(
        account_id=account_id
    ).first()
    for article in account.knowledge_articles:
        print(article)

<Knowledge(article_id='ef67d5f7-2d3b-4a6b-bcfe-bc65752f9860', title='How to Reserve a Spot for an Event')>
<Knowledge(article_id='f85fcf5e-2960-4194-9a5f-ee40fb81899f', title='What's Included in a CultPass Subscription')>
<Knowledge(article_id='9dd607e1-f1c6-42b0-9d20-e6b20cbd8fb4', title='How to Cancel or Pause a Subscription')>
<Knowledge(article_id='ce7d33c7-8829-4e7c-ac7f-983d4c290f34', title='How to Handle Login Issues?')>
<Knowledge(article_id='e08766f3-2d65-4ed0-8bee-a74b4cad26bf', title='How to Request a Refund')>
<Knowledge(article_id='46fbd95f-6f16-4204-b4b2-729addd2dbf4', title='How to Update Profile Information')>
<Knowledge(article_id='2fc9eb82-0d55-4552-8846-43e689569a98', title='How to Delete a CultPass Account')>
<Knowledge(article_id='c63c2648-a570-40a9-8e26-927f4ff79d97', title='Troubleshooting App Crashes')>
<Knowledge(article_id='afb8d3c9-c87d-44fd-aaee-31bd127dc1e6', title='QR Code Not Displaying or Scanning')>
<Knowledge(article_id='596104f7-4205-490f-a152-ec239ba

In [17]:
with get_session(engine) as session:
    users = session.query(udahub.User).all()
    for user in users:
        print(user)

<User(user_id='fb6a345c-50fb-421d-8a1f-1df728fce96c', user_name='Alice Kingsley', external_user_id='a4ab87')>


In [18]:
with get_session(engine) as session:
    user = session.query(udahub.User).filter_by(
        account_id=account_id,
        external_user_id=ticket_info["owner_id"],
    ).first()
    
    ticket:udahub.Ticket = user.tickets[0]
    for message in ticket.messages:
        print(message)

<TicketMessage(message_id='020aa492-73d9-4c15-8812-4794204204db', role='user', content='I can't log in to my Cultpass ...')>


## Rubric Validation

In [19]:
from sqlalchemy import inspect              
                                          
with get_session(engine) as session:
    # 1. Check required tables exist                                                                                                 
    inspector = inspect(engine)
    tables = inspector.get_table_names()                                                                                             
    required = ["accounts", "users", "tickets", "ticket_metadata", "ticket_messages", "knowledge"]                                 
    print("=== Tables ===")
    for t in required:
        status = "OK" if t in tables else "MISSING"
        print(f"  {t}: {status}")

    # 2. Count knowledge base articles
    article_count = session.query(udahub.Knowledge).count()
    print(f"\n=== Knowledge Base ===")
    print(f"  Total articles: {article_count} (need at least 14)")

    # 3. Check article categories
    articles = session.query(udahub.Knowledge).all()
    print(f"\n=== Article Topics ===")
    for a in articles:
        print(f"  - {a.title}")

    # 4. Sample data retrieval
    user = session.query(udahub.User).first()
    ticket = session.query(udahub.Ticket).first()
    print(f"\n=== Sample Retrieval ===")
    print(f"  User: {user}")
    print(f"  Ticket: {ticket}")

=== Tables ===
  accounts: OK
  users: OK
  tickets: OK
  ticket_metadata: OK
  ticket_messages: OK
  knowledge: OK

=== Knowledge Base ===
  Total articles: 17 (need at least 14)

=== Article Topics ===
  - How to Reserve a Spot for an Event
  - What's Included in a CultPass Subscription
  - How to Cancel or Pause a Subscription
  - How to Handle Login Issues?
  - How to Request a Refund
  - How to Update Profile Information
  - How to Delete a CultPass Account
  - Troubleshooting App Crashes
  - QR Code Not Displaying or Scanning
  - How to Leave a Review for an Experience
  - Reporting Suspicious Account Activity
  - How to Upgrade to CultPass Premium
  - Experience Complaint or Quality Issue
  - Payment Method Management
  - How to Use CultPass Gift Cards
  - Accessibility and Special Accommodations
  - How to Refer a Friend to CultPass

=== Sample Retrieval ===
  User: <User(user_id='fb6a345c-50fb-421d-8a1f-1df728fce96c', user_name='Alice Kingsley', external_user_id='a4ab87')>
  T